# 07 — Métricas Microcirculatorias por Video

Agregar resultados de segmentación + espacio-tiempo en métricas
consensadas ESICM: TVD, PVD, PPV, MFI, HI.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path(os.environ.get("MICROCIRCULATION_DATA", "/content/drive/MyDrive/microcirculation"))

seg_df = pd.read_csv(DATA_PATH / "segmentation_summary.csv")
vel_df = pd.read_csv(DATA_PATH / "velocity_estimates.csv")

# Cargar métricas por frame para calcular PPV
SEG_DIR = DATA_PATH / "segmented"

PERFUSION_THRESHOLD_UM_S = 100.0  # vasos < umbral = no perfundidos


## 7.1 Calcular PPV automático desde velocidades por vaso

In [ ]:
def compute_ppv_from_velocity(vel_df_row: pd.Series,
                               threshold: float = PERFUSION_THRESHOLD_UM_S) -> float:
    """
    PPV = proporción de vasos perfundidos.
    Un vaso es perfundido si vel > threshold (µm/s).
    """
    vel = vel_df_row['vel_mean']
    if pd.isna(vel):
        return np.nan
    # Aproximación con estimación por frame; en pipeline completo se haría por vaso
    return 1.0 if vel > threshold else 0.0


# Para un pipeline completo necesitamos vel por vaso — aquí usamos la mediana del video
# como proxy. Con múltiples vasos segmentados esto se expande naturalmente.
vel_df['PPV_auto'] = vel_df['vel_mean'].apply(
    lambda v: 1.0 if not pd.isna(v) and v > PERFUSION_THRESHOLD_UM_S else 0.0
)


## 7.2 Calcular MFI — Microvascular Flow Index

In [ ]:
def classify_flow_mfi(vel: float) -> int:
    """
    Clasificación ESICM:
    0 = ausente, 1 = intermitente, 2 = lento, 3 = continuo
    Umbrales aproximados — ajustar con datos propios.
    """
    if pd.isna(vel): return 0
    if vel < 50:     return 0   # ausente
    if vel < 200:    return 1   # intermitente
    if vel < 600:    return 2   # lento
    return 3                    # continuo

vel_df['MFI_auto'] = vel_df['vel_mean'].apply(classify_flow_mfi)


## 7.3 Calcular HI — Heterogeneity Index

In [ ]:
# HI = desviación estándar de velocidades entre videos del mismo paciente
# (definición simplificada; en pipeline completo = SD entre vasos del mismo frame)
# Acá calculamos la variabilidad inter-frame como proxy

st_files = list((DATA_PATH / "spacetime").rglob("spacetime.npy"))
hi_records = []

for sf in st_files:
    diag = np.load(str(sf))
    # Variabilidad temporal como proxy de HI
    frame_means = diag.mean(axis=1)  # intensidad media por frame
    hi_records.append({
        "video" : sf.parent.name,
        "HI_auto": float(frame_means.std())
    })

hi_df = pd.DataFrame(hi_records)


## 7.4 Consolidar métricas automáticas por video

In [ ]:
auto_metrics = (
    seg_df
    .merge(vel_df[['video','vel_mean','vel_hough','vel_xcorr','PPV_auto','MFI_auto']],
           on='video', how='left')
    .merge(hi_df,  on='video', how='left')
)

auto_metrics.rename(columns={
    'TVD_auto'     : 'TVD_auto',
    'n_vessels_mean': 'n_vessels_auto',
    'vel_mean'     : 'vel_mean_auto',
}, inplace=True)

auto_metrics.to_csv(DATA_PATH / "auto_metrics.csv", index=False)
print("Métricas automáticas por video:")
print(auto_metrics[['video','TVD_auto','PPV_auto','MFI_auto',
                     'vel_mean_auto','HI_auto']].to_string())


## 7.5 Visualización de distribuciones automáticas

In [ ]:
metrics_to_plot = ['TVD_auto','PPV_auto','vel_mean_auto','HI_auto']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, metrics_to_plot):
    data = auto_metrics[col].dropna()
    ax.hist(data, bins=12, color='steelblue', alpha=0.8, edgecolor='white')
    ax.set_title(col); ax.set_xlabel(col.split('_')[0])
    ax.axvline(data.median(), color='tomato', linestyle='--',
               label=f"Mediana: {data.median():.2f}")
    ax.legend(fontsize=8)
plt.suptitle("Distribución de métricas automáticas", fontsize=13)
plt.tight_layout()
plt.savefig(DATA_PATH / "fig_07_auto_metrics.png", dpi=150)
plt.show()
